# Protein ingestion (UniProt + Pfam)

Goal:
- Retrieve reviewed UniProt proteins (human) with Pfam cross-references
- Create a labeled dataset for protein family classification (Pfam ID)
- Save raw TSV and processed CSV under `data/`

In [1]:
from pathlib import Path
import time
import json
import re
from io import StringIO

import requests
import pandas as pd
import numpy as np
import yaml

ROOT = Path.cwd().parents[0]
DATA = ROOT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
REPORTS = ROOT / "reports"

for p in [RAW, PROCESSED, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

cfg_path = ROOT / "configs" / "config.yaml"
with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

SEED = int(cfg["project"]["random_seed"])
N_FAM = int(cfg["protein"]["n_families"])
PER_FAM = int(cfg["protein"]["per_family"])
MAX_LEN = int(cfg["protein"]["max_len_aa"])

(SEED, N_FAM, PER_FAM, MAX_LEN)

(42, 10, 400, 1024)

## UniProt fetch helper (TSV + pagination)

Query:
- reviewed:true
- organism_id:9606
- database:pfam

In [2]:
UNIPROT_BASE = "https://rest.uniprot.org/uniprotkb/search"

def fetch_uniprot_tsv(
    query: str,
    fields: list[str],
    out_path: Path,
    max_records: int = 20000,
    batch_size: int = 500,
    polite_sleep: float = 0.25,
) -> pd.DataFrame:
    """
    Fetch UniProtKB search results as TSV with pagination.
    Saves raw TSV to out_path and returns a DataFrame.
    """
    params = {
        "query": query,
        "format": "tsv",
        "fields": ",".join(fields),
        "size": batch_size,
    }

    chunks = []
    next_url = UNIPROT_BASE
    downloaded = 0

    while True:
        r = requests.get(next_url, params=params, timeout=90)
        r.raise_for_status()

        text = r.text
        lines = text.splitlines()
        if len(lines) <= 1:
            break

        # keep header only once
        if downloaded == 0:
            chunks.append(text if text.endswith("\n") else text + "\n")
        else:
            chunks.append("\n".join(lines[1:]) + "\n")

        downloaded += (len(lines) - 1)

        if downloaded >= max_records:
            break

        link = r.headers.get("Link", "")
        if 'rel="next"' not in link:
            break

        next_url = link.split("<")[1].split(">")[0]
        params = None  # cursor baked into next_url
        time.sleep(polite_sleep)

    raw_text = "".join(chunks)
    out_path.write_text(raw_text)

    return pd.read_csv(StringIO(raw_text), sep="\t")

## Download (or load cached) raw UniProt TSV

In [3]:
query = "reviewed:true AND organism_id:9606 AND database:pfam"

fields = [
    "accession",
    "protein_name",
    "sequence",
    "length",
    "xref_pfam",
    "organism_name",
]

raw_out = RAW / "uniprot_human_reviewed_pfam.tsv"

if raw_out.exists():
    df_raw = pd.read_csv(raw_out, sep="\t")
else:
    df_raw = fetch_uniprot_tsv(
        query=query,
        fields=fields,
        out_path=raw_out,
        max_records=20000,
        batch_size=500
    )

df_raw.shape

(19021, 6)

## Normalize columns + derive single-label Pfam ID

In [4]:
df = df_raw.copy()

# Normalize columns defensively
col_map = {}
for c in df.columns:
    cl = c.lower()
    if c in ["Entry", "Accession"] or "accession" in cl:
        col_map[c] = "accession"
    elif "sequence" in cl:
        col_map[c] = "sequence"
    elif "length" in cl:
        col_map[c] = "length"
    elif "pfam" in cl:
        col_map[c] = "pfam"

df = df.rename(columns=col_map)

needed = ["accession", "sequence", "length", "pfam"]
missing = [c for c in needed if c not in df.columns]
assert not missing, f"Missing columns: {missing}. Columns present: {df.columns.tolist()}"

def first_pfam(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"(PF\d{5})", str(val))
    return m.group(1) if m else np.nan

df["pfam_label"] = df["pfam"].apply(first_pfam)

df["length"] = pd.to_numeric(df["length"], errors="coerce")
df = df.dropna(subset=["accession", "sequence", "length", "pfam_label"]).copy()
df = df[df["length"] <= MAX_LEN].copy()

df.shape, df[["accession", "pfam_label", "length"]].head(3)

((16802, 7),
     accession pfam_label  length
 0      Q69383    PF15695     105
 1  A0A0C5B5G6    PF21945      16
 2      O42043    PF00517     560)

## Select top families + sample per family + de-duplicate + save processed CSV

Rules:
- Select top `n_families` by frequency in the cleaned dataset
- For each selected family, sample `min(per_family, available)` sequences
- Remove exact duplicate (sequence, family) rows deterministically

In [5]:
fam_counts = df["pfam_label"].value_counts()
top_fams = fam_counts.head(N_FAM).index.tolist()

df_top = df[df["pfam_label"].isin(top_fams)].copy()

# deterministic per-family sampling (no groupby.apply warning)
parts = []
for fam in top_fams:
    sub = df_top[df_top["pfam_label"] == fam]
    n_take = min(PER_FAM, len(sub))
    parts.append(sub.sample(n=n_take, random_state=SEED))

df_sampled = pd.concat(parts, ignore_index=True)

df_sampled = df_sampled.rename(columns={"pfam_label": "family"})
df_sampled = df_sampled[["accession", "sequence", "length", "family"]].copy()

# de-dup (sequence, family)
before = int(df_sampled.shape[0])
df_sampled = df_sampled.drop_duplicates(subset=["sequence", "family"], keep="first").reset_index(drop=True)
after = int(df_sampled.shape[0])
removed = before - after

out_csv = PROCESSED / f"protein_uniprot_pfam_top{N_FAM}_per{PER_FAM}.csv"
df_sampled.to_csv(out_csv, index=False)

out_csv, df_sampled.shape, removed

(PosixPath('/Users/saturnine/Projects/bio-seq-lm-capstone/data/processed/protein_uniprot_pfam_top10_per400.csv'),
 (2293, 4),
 16)

In [6]:
# integrity checks
assert df_sampled["accession"].isna().sum() == 0
assert df_sampled["sequence"].isna().sum() == 0
assert df_sampled["family"].isna().sum() == 0
assert df_sampled["length"].isna().sum() == 0

# sequence length consistency
assert (df_sampled["sequence"].str.len() == df_sampled["length"]).all(), "sequence length mismatch vs length column"

# duplicates after de-dup
dup_accessions = int(df_sampled["accession"].duplicated().sum())
dup_seq_family = int(df_sampled.duplicated(subset=["sequence", "family"]).sum())

dup_accessions, dup_seq_family

(0, 0)

In [7]:
summary = {
    "n_raw": int(df_raw.shape[0]),
    "n_clean": int(df.shape[0]),
    "n_sampled": int(df_sampled.shape[0]),
    "n_families": int(df_sampled["family"].nunique()),
    "top_families_selected": top_fams,
    "family_counts": df_sampled["family"].value_counts().to_dict(),
    "families": df_sampled["family"].value_counts().index.tolist(),
    "max_len_aa": int(MAX_LEN),
    "per_family_target": int(PER_FAM),
    "sampling_rule": "top_n_families_by_frequency; per_family = min(PER_FAM, available)",
    "seed": int(SEED),
    "uniprot_query": query,
    "dedup": {
        "removed_duplicate_sequence_family": int(removed),
        "dup_accessions_after": int(dup_accessions),
        "dup_sequence_family_after": int(dup_seq_family),
    },
}

summary_path = REPORTS / "protein_ingest_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

summary_path

PosixPath('/Users/saturnine/Projects/bio-seq-lm-capstone/reports/protein_ingest_summary.json')